# LAB 04 - Text Classification

Chạy từng cell theo thứ tự. Notebook dùng dataset CSV do bạn chọn, không ghi kết quả thay bạn. Train/validation/test được chia trước khi fit TF-IDF để tránh data leakage.

## 1. Imports và cấu hình
Chọn đường dẫn và tên cột văn bản/nhãn trong cell dưới đây.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab04':
    LAB_DIR = LAB_DIR / 'lab04'
sys.path.insert(0, str(LAB_DIR))
from preprocessing import normalize_text, split_dataset

DATA_PATH = LAB_DIR / 'data.csv'  # TODO: chọn dataset CSV.
TEXT_COLUMN = 'text'              # TODO: sửa theo tên cột trong dataset.
LABEL_COLUMN = 'label'            # TODO: sửa theo tên cột trong dataset.
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20  # Tỷ lệ train/validation/test = 60/20/20.
RANDOM_STATE = 42
POSITIVE_LABEL = None             # TODO: đặt nhãn positive nếu cần threshold binary.

candidate_models = {}
results_rows = []


## 2. Chọn và đọc dataset
W4 gợi ý 20 Newsgroups, AG News, IMDb, SMS Spam Collection, TREC-6, SST-2, Yelp Polarity, DBPedia, SNLI hoặc Quora Question Pairs.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Đặt file dataset tại {DATA_PATH} hoặc sửa DATA_PATH trong cell cấu hình.'
    )

raw_df = pd.read_csv(DATA_PATH)
if TEXT_COLUMN not in raw_df or LABEL_COLUMN not in raw_df:
    raise KeyError(
        f'Không tìm thấy cột {TEXT_COLUMN!r} hoặc {LABEL_COLUMN!r}. '
        f'Các cột hiện có: {list(raw_df.columns)}'
    )
display(pd.DataFrame([{
    'Tệp': DATA_PATH.name,
    'Số dòng ban đầu': len(raw_df),
    'Cột văn bản': TEXT_COLUMN,
    'Cột nhãn': LABEL_COLUMN,
}]))


## 3. Dataset inspection
Kiểm tra missing values, document trùng, độ dài văn bản và class distribution trước khi train.

In [ ]:
data = raw_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
quality_table = pd.DataFrame([{
    'Missing text': int(data[TEXT_COLUMN].isna().sum()),
    'Missing labels': int(data[LABEL_COLUMN].isna().sum()),
    'Duplicate texts': int(data[TEXT_COLUMN].duplicated().sum()),
    'Classes': int(data[LABEL_COLUMN].nunique(dropna=True)),
}])
display(quality_table)


In [ ]:
data = data.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
data[TEXT_COLUMN] = data[TEXT_COLUMN].map(normalize_text)
data[LABEL_COLUMN] = data[LABEL_COLUMN].astype(str)

label_counts_per_text = data.groupby(TEXT_COLUMN)[LABEL_COLUMN].nunique()
conflicting_texts = set(label_counts_per_text[label_counts_per_text > 1].index)
conflicting_count = len(conflicting_texts)
duplicate_count_before = int(data[TEXT_COLUMN].duplicated().sum())

# Loại văn bản trùng nhãn mâu thuẫn, rồi giữ một bản cho mỗi văn bản để tránh leakage.
if conflicting_texts:
    data = data[~data[TEXT_COLUMN].isin(conflicting_texts)].copy()
data = data.drop_duplicates(subset=[TEXT_COLUMN], keep='first').reset_index(drop=True)

display(pd.DataFrame([{
    'Văn bản có nhãn mâu thuẫn đã bỏ': conflicting_count,
    'Văn bản trùng đã bỏ': duplicate_count_before,
    'Số dòng còn lại': len(data),
}]))


In [ ]:
data['document_length'] = data[TEXT_COLUMN].map(lambda text: len(text.split()))
length_table = data.groupby(LABEL_COLUMN)['document_length'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).reset_index()
length_table = length_table.rename(columns={
    LABEL_COLUMN: 'Class',
    'count': 'Số mẫu',
    'mean': 'Độ dài trung bình',
    'median': 'Trung vị độ dài',
    'min': 'Ngắn nhất',
    'max': 'Dài nhất',
})
display(length_table)


In [ ]:
class_distribution = data[LABEL_COLUMN].value_counts().rename_axis('Class').reset_index(name='Count')
class_distribution['Percentage'] = (
    100 * class_distribution['Count'] / class_distribution['Count'].sum()
)
display(class_distribution)


In [ ]:
import matplotlib.pyplot as plt

ax = class_distribution.set_index('Class')['Count'].plot(kind='bar')
ax.set_xlabel('Class')
ax.set_ylabel('\u0053\u1ed1 m\u1eabu')
ax.set_title('Ph\u00e2n ph\u1ed1i class')
plt.tight_layout()
plt.show()


## 4. Train/validation/test split
Chia dữ liệu có stratification trước. TF-IDF chỉ được fit bên trong pipeline trên train; validation/test chỉ được transform.

In [ ]:
texts = data[TEXT_COLUMN].tolist()
labels = data[LABEL_COLUMN].tolist()
(
    X_train, X_validation, X_test,
    y_train, y_validation, y_test,
) = split_dataset(
    texts,
    labels,
    test_size=TEST_SIZE,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)
split_table = pd.DataFrame([
    {'Tập': 'Train', 'Số mẫu': len(X_train)},
    {'Tập': 'Validation', 'Số mẫu': len(X_validation)},
    {'Tập': 'Test', 'Số mẫu': len(X_test)},
])
display(split_table)


In [ ]:
split_distribution = []
for split_name, split_labels in (
    ('Train', y_train),
    ('Validation', y_validation),
    ('Test', y_test),
):
    counts = pd.Series(split_labels).value_counts(normalize=True)
    for label, proportion in counts.items():
        split_distribution.append({
            'Tập': split_name,
            'Class': label,
            'Tỷ lệ': proportion,
        })
display(pd.DataFrame(split_distribution))


## 5. Metrics helper
Dùng macro precision/recall/F1 để các lớp cùng có trọng số khi class distribution lệch.

In [ ]:
def classification_metrics(model, features, true_labels):
    predicted = model.predict(features)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted, average='macro', zero_division=0
    )
    return {
        'accuracy': accuracy_score(true_labels, predicted),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }


def add_validation_result(name, representation, classifier, model, features,
                          training_seconds, vocabulary_size=None, ngram_range=''):
    metrics = classification_metrics(model, features, y_validation)
    row = {
        'model': name,
        'representation': representation,
        'classifier': classifier,
        'split': 'validation',
        **metrics,
        'training_time_seconds': training_seconds,
        'vocabulary_size': vocabulary_size,
        'ngram_range': ngram_range,
        'threshold': '',
        'notes': '',
    }
    results_rows.append(row)
    return row


## 6. Baseline - TF-IDF + Logistic Regression
Pipeline giúp vectorizer chỉ học vocabulary/IDF từ train.

In [ ]:
tfidf_lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=1,
        stop_words=None,
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
started = time.perf_counter()
tfidf_lr_pipeline.fit(X_train, y_train)
tfidf_lr_seconds = time.perf_counter() - started
tfidf_vocabulary_size = len(tfidf_lr_pipeline.named_steps['tfidf'].vocabulary_)

candidate_models['TF-IDF + Logistic Regression'] = {
    'model': tfidf_lr_pipeline,
    'validation_features': X_validation,
    'test_features': X_test,
    'representation': 'TF-IDF unigram',
    'classifier': 'Logistic Regression',
}
baseline_validation = add_validation_result(
    'TF-IDF + Logistic Regression',
    'TF-IDF unigram',
    'Logistic Regression',
    tfidf_lr_pipeline,
    X_validation,
    tfidf_lr_seconds,
    vocabulary_size=tfidf_vocabulary_size,
    ngram_range='(1, 1)',
)
display(pd.DataFrame([baseline_validation]))


## 7. Threshold - binary classification
Chỉ áp dụng khi có đúng hai class. Tăng threshold thường đổi precision/recall; ghi kết quả của dataset.

In [ ]:
threshold_rows = []
classes = tfidf_lr_pipeline.named_steps['classifier'].classes_
if len(classes) == 2:
    positive_label = POSITIVE_LABEL if POSITIVE_LABEL is not None else classes[1]
    if positive_label not in classes:
        raise ValueError(f'Positive label {positive_label!r} không nằm trong classes.')
    positive_index = list(classes).index(positive_label)
    probabilities = tfidf_lr_pipeline.predict_proba(X_validation)[:, positive_index]
    negative_label = classes[1 - positive_index]

    for threshold in (0.5, 0.8):
        threshold_predictions = np.where(
            probabilities >= threshold, positive_label, negative_label
        )
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_validation,
            threshold_predictions,
            average='binary',
            pos_label=positive_label,
            zero_division=0,
        )
        threshold_rows.append({
            'Threshold': threshold,
            'Positive label': positive_label,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
        })
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y_validation,
            threshold_predictions,
            average='macro',
            zero_division=0,
        )
        results_rows.append({
            'model': 'TF-IDF + Logistic Regression',
            'representation': 'TF-IDF unigram',
            'classifier': 'Logistic Regression',
            'split': 'validation',
            'accuracy': accuracy_score(y_validation, threshold_predictions),
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'f1_macro': f1_macro,
            'training_time_seconds': tfidf_lr_seconds,
            'vocabulary_size': tfidf_vocabulary_size,
            'ngram_range': '(1, 1)',
            'threshold': threshold,
            'notes': f'Positive label: {positive_label}',
        })
else:
    threshold_rows.append({
        'Threshold': 'N/A',
        'Ghi chú': 'Bài toán không phải binary classification.',
    })
display(pd.DataFrame(threshold_rows))


## 8. Representation - Average Word2Vec
Gensim cần được cài trước cell này. Chỉ train embedding trên văn bản train để tránh dùng validation/test trong quá trình học.

In [ ]:
try:
    from gensim.models import Word2Vec
except ImportError as error:
    raise ImportError('Cài Gensim trước: pip install gensim') from error

w2v_vector_size = 100
w2v_window = 5
w2v_min_count = 2
w2v_epochs = 10
train_tokens = [text.split() for text in X_train]
started = time.perf_counter()
word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=w2v_vector_size,
    window=w2v_window,
    min_count=w2v_min_count,
    workers=1,
    sg=1,
    epochs=w2v_epochs,
    seed=RANDOM_STATE,
)
word2vec_train_seconds = time.perf_counter() - started

def average_word2vec(texts, keyed_vectors):
    rows = []
    for text in texts:
        vectors = [
            keyed_vectors[token]
            for token in text.split()
            if token in keyed_vectors
        ]
        rows.append(
            np.mean(vectors, axis=0)
            if vectors
            else np.zeros(keyed_vectors.vector_size, dtype=np.float32)
        )
    return np.vstack(rows)

X_train_w2v = average_word2vec(X_train, word2vec_model.wv)
X_validation_w2v = average_word2vec(X_validation, word2vec_model.wv)
X_test_w2v = average_word2vec(X_test, word2vec_model.wv)


In [ ]:
w2v_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
started = time.perf_counter()
w2v_lr.fit(X_train_w2v, y_train)
w2v_lr_seconds = time.perf_counter() - started
candidate_models['Average Word2Vec + Logistic Regression'] = {
    'model': w2v_lr,
    'validation_features': X_validation_w2v,
    'test_features': X_test_w2v,
    'representation': 'Average Word2Vec',
    'classifier': 'Logistic Regression',
}
w2v_validation = add_validation_result(
    'Average Word2Vec + Logistic Regression',
    'Average Word2Vec',
    'Logistic Regression',
    w2v_lr,
    X_validation_w2v,
    word2vec_train_seconds + w2v_lr_seconds,
    vocabulary_size=len(word2vec_model.wv),
)
display(pd.DataFrame([w2v_validation]))


## 9. N-gram features - unigram và unigram + bigram
Fit từng vectorizer qua Pipeline trên train. So sánh vocabulary size, validation F1 và thời gian.

In [ ]:
tfidf_bigram_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        stop_words=None,
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
started = time.perf_counter()
tfidf_bigram_pipeline.fit(X_train, y_train)
tfidf_bigram_seconds = time.perf_counter() - started
tfidf_bigram_vocabulary_size = len(
    tfidf_bigram_pipeline.named_steps['tfidf'].vocabulary_
)
candidate_models['TF-IDF unigram + bigram + Logistic Regression'] = {
    'model': tfidf_bigram_pipeline,
    'validation_features': X_validation,
    'test_features': X_test,
    'representation': 'TF-IDF unigram + bigram',
    'classifier': 'Logistic Regression',
}
bigram_validation = add_validation_result(
    'TF-IDF unigram + bigram + Logistic Regression',
    'TF-IDF unigram + bigram',
    'Logistic Regression',
    tfidf_bigram_pipeline,
    X_validation,
    tfidf_bigram_seconds,
    vocabulary_size=tfidf_bigram_vocabulary_size,
    ngram_range='(1, 2)',
)
display(pd.DataFrame([{
    'N-gram range': '(1, 1)',
    'Vocabulary size': tfidf_vocabulary_size,
    'Validation F1 macro': baseline_validation['f1_macro'],
    'Training time (s)': tfidf_lr_seconds,
}, {
    'N-gram range': '(1, 2)',
    'Vocabulary size': tfidf_bigram_vocabulary_size,
    'Validation F1 macro': bigram_validation['f1_macro'],
    'Training time (s)': tfidf_bigram_seconds,
}]))


## 10. Word order và phủ định
Unigram TF-IDF và Average Word2Vec phụ thuộc vào tập từ, không giữ vị trí từ. Bigram có thể giữ các cụm như not good.

In [ ]:
order_pair = ['not good', 'good not']
unigram_order_vectors = TfidfVectorizer(ngram_range=(1, 1)).fit_transform(order_pair)
bigram_order_vectors = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(order_pair)
average_order_vectors = average_word2vec(order_pair, word2vec_model.wv)

word_order_table = pd.DataFrame([
    {
        'Representation': 'TF-IDF unigram',
        'Cosine(not good, good not)': cosine_similarity(
            unigram_order_vectors[0], unigram_order_vectors[1]
        )[0, 0],
    },
    {
        'Representation': 'TF-IDF unigram + bigram',
        'Cosine(not good, good not)': cosine_similarity(
            bigram_order_vectors[0], bigram_order_vectors[1]
        )[0, 0],
    },
    {
        'Representation': 'Average Word2Vec',
        'Cosine(not good, good not)': cosine_similarity(
            average_order_vectors[0].reshape(1, -1),
            average_order_vectors[1].reshape(1, -1),
        )[0, 0],
    },
])
display(word_order_table)

negation_pair = ['this movie is good', 'this movie is not good']
negation_unigram = TfidfVectorizer(ngram_range=(1, 1)).fit_transform(negation_pair)
negation_bigram = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(negation_pair)
negation_average = average_word2vec(negation_pair, word2vec_model.wv)
negation_table = pd.DataFrame([
    {
        'Representation': 'TF-IDF unigram',
        'Cosine(good, not good)': cosine_similarity(
            negation_unigram[0], negation_unigram[1]
        )[0, 0],
    },
    {
        'Representation': 'TF-IDF unigram + bigram',
        'Cosine(good, not good)': cosine_similarity(
            negation_bigram[0], negation_bigram[1]
        )[0, 0],
    },
    {
        'Representation': 'Average Word2Vec',
        'Cosine(good, not good)': cosine_similarity(
            negation_average[0].reshape(1, -1),
            negation_average[1].reshape(1, -1),
        )[0, 0],
    },
])
display(negation_table)


## 11. Optional model - TF-IDF + Linear SVM
Chạy nếu còn thời gian; dùng cùng train/validation split và vectorizer fit trên train.

In [ ]:
tfidf_svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=1,
        stop_words=None,
    )),
    ('classifier', LinearSVC(random_state=RANDOM_STATE)),
])
started = time.perf_counter()
tfidf_svm_pipeline.fit(X_train, y_train)
tfidf_svm_seconds = time.perf_counter() - started
tfidf_svm_vocabulary_size = len(
    tfidf_svm_pipeline.named_steps['tfidf'].vocabulary_
)
candidate_models['TF-IDF + Linear SVM'] = {
    'model': tfidf_svm_pipeline,
    'validation_features': X_validation,
    'test_features': X_test,
    'representation': 'TF-IDF unigram',
    'classifier': 'Linear SVM',
}
svm_validation = add_validation_result(
    'TF-IDF + Linear SVM',
    'TF-IDF unigram',
    'Linear SVM',
    tfidf_svm_pipeline,
    X_validation,
    tfidf_svm_seconds,
    vocabulary_size=tfidf_svm_vocabulary_size,
    ngram_range='(1, 1)',
)
display(pd.DataFrame([svm_validation]))


## 12. So sánh trên validation và chọn model
Chọn model theo macro F1 trên validation. Chỉ dùng test để đánh giá model được chọn sau bước này.

In [ ]:
validation_table = pd.DataFrame(results_rows)
validation_table = validation_table[
    (validation_table['split'] == 'validation')
    & (validation_table['threshold'].fillna('') == '')
]
display(validation_table.sort_values('f1_macro', ascending=False))

best_name = validation_table.sort_values(
    'f1_macro', ascending=False
).iloc[0]['model']
best_entry = candidate_models[best_name]
best_test_predictions = best_entry['model'].predict(best_entry['test_features'])
best_test_metrics = classification_metrics(
    best_entry['model'], best_entry['test_features'], y_test
)
test_result = {
    'model': best_name,
    'representation': best_entry['representation'],
    'classifier': best_entry['classifier'],
    'split': 'test',
    **best_test_metrics,
    'training_time_seconds': np.nan,
    'vocabulary_size': np.nan,
    'ngram_range': '',
    'threshold': '',
    'notes': 'Model được chọn theo validation macro F1.',
}
results_rows.append(test_result)
display(pd.DataFrame([test_result]))


## 13. Confusion matrix
Vẽ confusion matrix cho test set của model được chọn theo validation, rồi lưu ảnh theo deliverable.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

labels_in_order = sorted(pd.Series(y_test).unique())
matrix = confusion_matrix(
    y_test,
    best_test_predictions,
    labels=labels_in_order,
)
display(pd.DataFrame(
    matrix,
    index=[f'Thật: {label}' for label in labels_in_order],
    columns=[f'Dự đoán: {label}' for label in labels_in_order],
))

display_plot = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=labels_in_order,
)
display_plot.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.tight_layout()
CONFUSION_PATH = LAB_DIR / 'confusion_matrix.png'
plt.savefig(CONFUSION_PATH, dpi=160, bbox_inches='tight')
plt.show()
plt.close()


## 14. Error analysis - ví dụ phân loại
Chọn tối đa 3 ví dụ đúng và 3 ví dụ sai từ test để phân tích trong error_analysis.md.

In [ ]:
test_examples = pd.DataFrame({
    'Text': X_test,
    'True label': y_test,
    'Predicted label': best_test_predictions,
})
correct_examples = test_examples[
    test_examples['True label'] == test_examples['Predicted label']
].head(3)
incorrect_examples = test_examples[
    test_examples['True label'] != test_examples['Predicted label']
].head(3)
display(correct_examples)
display(incorrect_examples)


## 15. Inspect model weights - TF-IDF + Logistic Regression
Xem feature có trọng số dương/âm. Với binary classification, hệ số dương ủng hộ class thứ hai; với multiclass, mỗi hàng hệ số ứng với một class.

In [ ]:
lr_classifier = tfidf_lr_pipeline.named_steps['classifier']
feature_names = tfidf_lr_pipeline.named_steps['tfidf'].get_feature_names_out()
weight_rows = []

if lr_classifier.coef_.shape[0] == 1:
    weights = lr_classifier.coef_[0]
    positive_class = lr_classifier.classes_[1]
    negative_class = lr_classifier.classes_[0]
    for index in np.argsort(weights)[-10:][::-1]:
        weight_rows.append({
            'Class': positive_class,
            'Feature': feature_names[index],
            'Weight': weights[index],
        })
    for index in np.argsort(weights)[:10]:
        weight_rows.append({
            'Class': negative_class,
            'Feature': feature_names[index],
            'Weight': weights[index],
        })
else:
    for class_index, class_label in enumerate(lr_classifier.classes_):
        weights = lr_classifier.coef_[class_index]
        for index in np.argsort(weights)[-10:][::-1]:
            weight_rows.append({
                'Class': class_label,
                'Feature': feature_names[index],
                'Weight': weights[index],
            })

weights_table = pd.DataFrame(weight_rows)
display(weights_table)


## 16. Lưu kết quả
Chạy cell này sau khi đã chạy các thí nghiệm bạn chọn. Confusion matrix được lưu riêng thành PNG.

In [ ]:
RESULTS_PATH = LAB_DIR / 'results.csv'
pd.DataFrame(results_rows).to_csv(RESULTS_PATH, index=False)
display(pd.DataFrame([{
    'Results CSV': RESULTS_PATH.name,
    'Confusion matrix': CONFUSION_PATH.name,
    'Số dòng kết quả': len(results_rows),
}]))
